# 세 텍스트의 encoding 설명력 비교 (탐색, 피험자 1)
기준은 Notion 「텍스트 3종의 임베딩 구성 설계」와 「RQ1 판정 설계 초안」에 사전 기록. 코드는 `src/fmri/text_compare.py`.
Drive에 필요한 것: `fmri_capstone/fmri/roi_betas/`(이미 있음), `fmri_capstone/embeddings_v2/`(전체 실행 결과, `ctx_ids.json` 포함), `fmri_capstone/colab_bundle/`(captions, labels_pass0).
탐색 단계이므로 결론이 아니라 곡선을 보는 것이다. 확인은 shared1000에서 따로 한다. 모델 하나당 수십 분 걸릴 수 있다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/fmri_capstone'
!rm -rf /content/repo && git clone -q https://github.com/geon-2/fmri-imagery-context /content/repo
!pip -q install scipy
!cd /content/repo/src/fmri && PYTHONPATH=/content/repo/src/labeling python3 text_compare.py --selftest | tail -2

## MPNet 실행 (V3 항목별 6블록은 차원이 커서 느리다. 먼저 `--skip ctx_V3`로 돌리고 나중에 따로 봐도 된다)

In [ ]:
!cd /content/repo/src/fmri && PYTHONPATH=/content/repo/src/labeling python3 -u text_compare.py \
  --fmri-dir {BASE}/fmri/roi_betas --emb-dir {BASE}/embeddings_v2 --bundle-dir {BASE}/colab_bundle \
  --out-dir {BASE}/text_compare --model MPNet --skip ctx_V3

## CLIP 실행 (V2의 맥락 문자열은 46%가 77토큰에서 잘린다는 점을 해석할 때 고려)

In [ ]:
!cd /content/repo/src/fmri && PYTHONPATH=/content/repo/src/labeling python3 -u text_compare.py \
  --fmri-dir {BASE}/fmri/roi_betas --emb-dir {BASE}/embeddings_v2 --bundle-dir {BASE}/colab_bundle \
  --out-dir {BASE}/text_compare --model CLIP --skip ctx_V3

## 그림으로 보기

In [ ]:
import json, numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), sharey=True)
for ax, m in zip(axes, ['MPNet', 'CLIP']):
    R = json.load(open(f'{BASE}/text_compare/text_compare_{m}.json'))
    names = ['orig_P1', 'obj_P1', 'ctx_P1', 'obj_P1+ctx_P1', 'orig_V2', 'obj_V2', 'ctx_V2', 'obj_V2+ctx_V2']
    names = [n for n in names if n in R['P']]
    y = np.arange(len(names))
    P = [R['P'][n]['P'] for n in names]; lo = [R['P'][n]['P'] - R['P'][n]['ci'][0] for n in names]; hi = [R['P'][n]['ci'][1] - R['P'][n]['P'] for n in names]
    ax.barh(y + 0.2, P, 0.4, xerr=[lo, hi], color='#4c72b0', label='real text')
    sh = [R['P'].get(n + '_shuf', {'P': np.nan})['P'] for n in names]
    ax.barh(y - 0.2, sh, 0.4, color='#bbbbbb', label='shuffled (same item counts)')
    ax.set_yticks(y); ax.set_yticklabels(names); ax.invert_yaxis(); ax.axvline(0, color='k', lw=.8)
    ax.set_title(f'{m}: P = median voxel r (R2, R3 mean), subject 1, {R["n_images"]} images'); ax.set_xlabel('median voxel r (cross-validated)')
    ax.legend(loc='lower right')
plt.tight_layout(); plt.savefig(BASE + '/text_compare/p_bars.png', dpi=150); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6), sharex=True)
keys = [('ctx_unique_beyond_obj_adj', 'context unique beyond object (real - shuffled context)'), ('ctx_unique_beyond_obj_raw', 'context added to object (raw)'),
        ('ctx_above_shuffle', 'context vs shuffled'), ('obj_above_shuffle', 'object vs shuffled'), ('orig_above_shuffle', 'original vs shuffled'), ('joint_vs_orig', 'object+context minus original')]
for ax, m in zip(axes, ['MPNet', 'CLIP']):
    R = json.load(open(f'{BASE}/text_compare/text_compare_{m}.json'))
    rows = [(f'{lab} [{v}]', R['contrasts'][v][k]) for k, lab in keys for v in ('P1', 'V2') if R['contrasts'][v][k]]
    y = np.arange(len(rows))
    for i, (lab, c) in enumerate(rows):
        ax.errorbar(c['diff'], i, xerr=[[c['diff'] - c['ci'][0]], [c['ci'][1] - c['diff']]], fmt='o', color='#4c72b0' if '[P1]' in lab else '#dd8452')
    ax.set_yticks(y); ax.set_yticklabels([r[0] for r in rows], fontsize=8); ax.invert_yaxis()
    ax.axvline(0, color='k', lw=.8); ax.axvline(0.01, color='red', ls='--', lw=.8)
    ax.set_title(f"{m}: contrasts (95% CI). red = effect floor 0.01\nverdict: {R['ctx_unique_claim']['verdict']}"); ax.set_xlabel('difference in P')
plt.tight_layout(); plt.savefig(BASE + '/text_compare/contrasts.png', dpi=150); plt.show()

## shared1000 확인 (탐색을 끝낸 뒤에만; 같은 세트·같은 판정 규칙)
피험자 1 기준. 탐색 표본(학습용 라벨 있는 이미지 전체)으로 적합하고 shared1000 1,000장에서만 평가한다. ctx_V3는 건너뛴다. 다른 피험자는 `--fmri-dir`와 `--out-dir`만 바꿔 같은 방식으로 돌린다.

In [ ]:
for MODEL in ['MPNet', 'CLIP']:
    !cd /content/repo/src/fmri && PYTHONPATH=/content/repo/src/labeling python3 -u text_compare.py \
      --fmri-dir {BASE}/fmri/roi_betas --emb-dir {BASE}/embeddings_v2 --bundle-dir {BASE}/colab_bundle \
      --out-dir {BASE}/text_compare --model {MODEL} --skip ctx_V3 --confirm